In [41]:
%reload_ext autoreload

%autoreload 2

%reload_ext dotenv
%dotenv

In [42]:
from mlde_analysis.furflex_default_params import *

In [43]:
import dask
import dask.array
from dask.distributed import Client
import functools
import math
import string

import IPython
from IPython.display import HTML
import matplotlib
from matplotlib import animation
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import os
import cf_xarray

from mlde_utils import cp_model_rotated_pole, VariableMetadata
from mlde_analysis import DERIVED_DATA, plot_map, projection_from_da
from mlde_analysis.display import pretty_table, VAR_RANGES
from mlde_analysis.data import si_to_mmhour
from mlde_analysis.furflex_data import open_dataset_predictors_split

In [44]:
client = Client()
client

/home/vf20964/furflex/code/mlde-analysis/.pixi/envs/dev/lib/python3.12/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 45297 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:45297/status,
Dashboard: http://127.0.0.1:45297/status,Workers: 4
Total threads: 16,Total memory: 15.35 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:45611,Workers: 0
Dashboard: http://127.0.0.1:45297/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:43029,Total threads: 4
Dashboard: http://127.0.0.1:45519/status,Memory: 3.84 GiB
Nanny: tcp://127.0.0.1:33157,


In [45]:
matplotlib.rcParams['figure.dpi'] = 100

In [46]:
IPython.display.Markdown(desc)


Describe in more detail the models being compared


In [47]:
%reload_ext mlde_analysis.furflex_magics 
EVAL_DS, MODELS, TARGET_DAS, PRED_DAS, VAR_DAS, MODELLABEL2SPEC = %load_eval_data
EVAL_DS["CPM"]

<xarray.Dataset> Size: 104MB
Dimensions:                       (model: 1, sample_id: 1, ensemble_member: 1,
                                   projection_y_coordinate: 128,
                                   projection_x_coordinate: 128, time: 792,
                                   bnds: 2)
Coordinates:
  * model                         (model) object 8B 'e0025'
  * ensemble_member               (ensemble_member) object 8B 'r001i1p00000'
  * projection_y_coordinate       (projection_y_coordinate) float64 1kB -2.75...
  * projection_x_coordinate       (projection_x_coordinate) float64 1kB 5.75e...
  * time                          (time) object 6kB 1989-12-01 00:30:00 ... 2...
    latitude                      (projection_y_coordinate, projection_x_coordinate) float64 131kB dask.array<chunksize=(128, 128), meta=np.ndarray>
    longitude                     (projection_y_coordinate, projection_x_coordinate) float64 131kB dask.array<chunksize=(128, 128), meta=np.ndarray>
    time_period                   (time) <U8 25kB 'historic' ... 'future'
    dec_adjusted_year             (time) int64 6kB 1990 1990 1990 ... 2065 2065
    stratum                       (time) <U12 38kB 'historic DJF' ... 'future...
    tp_season_year                (time) <U17 54kB 'historic DJF 1990' ... 'f...
Dimensions without coordinates: sample_id, bnds
Data variables:
    pred_pr                       (model, sample_id, ensemble_member, projection_y_coordinate, projection_x_coordinate, time) float32 52MB dask.array<chunksize=(1, 1, 1, 128, 128, 792), meta=np.ndarray>
    target_pr                     (ensemble_member, time, projection_y_coordinate, projection_x_coordinate) float32 52MB dask.array<chunksize=(1, 792, 128, 128), meta=np.ndarray>
    projection_x_coordinate_bnds  (projection_x_coordinate, bnds) float64 2kB dask.array<chunksize=(128, 2), meta=np.ndarray>
    projection_y_coordinate_bnds  (projection_y_coordinate, bnds) float64 2kB dask.array<chunksize=(128, 2), meta=np.ndarray>
    transverse_mercator           int32 4B ...
Attributes:
    grid_mapping:   rotated_latitude_longitude
    standard_name:  pred_pr
    units:          kg m-2 s-1

In [91]:
predictors_ds = open_dataset_predictors_split(dataset_configs["CPM"], split)

In [102]:
# em_time = ("01", cftime.Datetime360Day(1993, 8, 1, 12))

# plot_ds = EVAL_DS["CPM"].sel(time=em_time[1], method="nearest").sel(ensemble_member=em_time[0]).sel(model=list(MODELS["CPM"].keys())[-1])
# plot_ds = ds
var = "pr"

nsamples = 3
nframes = 24

example_spec = examples_to_plot["CPM"]["v3"]

pred_ds = EVAL_DS["CPM"].sel(time=slice(*example_spec["times"]), **example_spec["query"]).isel(model=0)

example_predictors_ds = predictors_ds.sel(time=slice(*example_spec["times"]), **example_spec["query"])

thetas = [850, 700, 500, 250]

possible_variables = [("spechum", thetas, "Specific\nHumidity\n(multi-level)"), ("temp", thetas, "Temperature\n(multi-level)"), ("vort", thetas, "Vorticity\n(multi-level)"), ("psl", [""], "Sea-level pressure")]
variables = possible_variables # list(filter(lambda v: any([k.startswith(v[0]) for k in plot_ds.variables.keys()]), possible_variables))

full_variable_set = [f"{varclass}{level}" for varclass, levels, _title in variables for level in levels]


awidth = 0.12
offset_width = 0.009
stacked_awidth = (awidth + offset_width *3)
gap = (1 - stacked_awidth * 4)/(len(variables) - 1)
print(stacked_awidth*4 + gap*3)

fig = plt.figure(figsize=(6.5, 5.5), layout="constrained")

mosaic = [full_variable_set + [f"pred_pr {i}" for i in range(nsamples)] + ["AI"]]

per_subplot_kw = {f"pred_pr {i}": {"projection": projection_from_da(pred_ds.cf[f"pred_{var}"])} for i in range(nsamples)} | {v: {"projection": projection_from_da(predictors_ds.cf[v])} for v in full_variable_set}

axd = fig.subplot_mosaic(mosaic, per_subplot_kw=per_subplot_kw)



ax = axd["AI"]
# ax.axis("off")
ax.xaxis.set_major_locator(plt.NullLocator())
ax.yaxis.set_major_locator(plt.NullLocator())
ax.set_facecolor('black')
ax.text(0.5, 0.5, "CPMGEM\nhourly",
        ha='center', va='center', color="white", weight='bold', transform=ax.transAxes, clip_on=False)
ax.set_position([0.43, awidth+0.15, 0.15, 0.15])

output_arrows = [
    dict(
        xy=(1, 1),
        xytext=(0.5, 0),
        arrowprops=dict(facecolor='black', shrinkB=5, arrowstyle="fancy", connectionstyle="arc3,rad=-0.2"),
    ),
    dict(
        xy=(0.5, 1),
        xytext=(0.5, 0),
        arrowprops=dict(facecolor='black', shrinkB=0, arrowstyle="fancy"),
    ),
    dict(
        xy=(0, 1),
        xytext=(0.5, 0),
        arrowprops=dict(facecolor='black', shrinkB=5, arrowstyle="fancy", connectionstyle="arc3,rad=0.2"),
    ),
]
pr_quads = []
for sampleidx in range(nsamples):
    ax = axd[f"pred_pr {sampleidx}"]
    ax.set_position([(0.505-awidth/2)+(sampleidx-1)*(awidth+0.03), awidth/2, awidth, awidth])

    if sampleidx == 1:
        ax.text(0.5, -0.15, "High-resolution precipitation", fontsize="small", ha='center', va='center',transform=ax.transAxes)

    axd["AI"].annotate(
            '',
            xycoords=ax.transAxes,
            textcoords=axd["AI"].transAxes,
            **output_arrows[sampleidx],
        )
    
    if sampleidx == 1:
        pr_quad = plot_map(pred_da.isel(sample_id=0, time=0), ax=ax, style="pr_hourly")
        pr_quads.append(pr_quad)
    else:
        ax.axis("off")
    
arrows = [
    dict(
        xy=(0.5, 0.5),
        xytext=(0.5, 0.5),
        arrowprops=dict(facecolor='black', shrinkA=42, shrinkB=35, arrowstyle="simple"),#, connectionstyle="arc3,rad=0.2"),
    ),
    dict(
        xy=(0.5, 0.5),
        xytext=(0.85, -0.25),
        arrowprops=dict(facecolor='black', shrinkB=42, arrowstyle="simple"),#, connectionstyle="arc3,rad=0.2"),
    ),
    dict(
        xy=(0.5, 0.5),
        xytext=(0.5, -0.25),
        arrowprops=dict(facecolor='black', shrinkB=42, arrowstyle="simple"),#, connectionstyle="arc3,rad=-0.2"),
    ),
    dict(
        xy=(0.5, 0.5),
        xytext=(0.5, 0.5),
        arrowprops=dict(facecolor='black', shrinkA=30, shrinkB=37, arrowstyle="simple"),#, connectionstyle="arc3,rad=-0.2"),
    ),
]

for vi, (varclass, levels, vartitle) in enumerate(variables):
    variable_set = [f"{varclass}{level}" for level in levels]

    for i, var in enumerate(variable_set):
        ax = axd[var]
        var_plot_kwargs = {}
        if varclass in ["vorticity"]:
            var_plot_kwargs = {"center": 0}
        plot_map(example_predictors_ds[var].isel(time=0), ax=ax, style=None, **var_plot_kwargs)
        # plot_map(plot_ds["pr"].isel(time=0), ax=ax, style=None, **var_plot_kwargs)
        left = (stacked_awidth + gap) * vi + offset_width*i
        top = 0.5-offset_width*i
        if vi == 0 or vi == len(variables) - 1:
            top = top - 0.1
        ax.set_position([left, top, awidth, awidth])
        if i == 0:
            ax.set_title(vartitle, fontsize="small")
        if i == 0:#len(variable_set)-1:
            axd["AI"].annotate(
                '',
                xycoords=axd["AI"].transAxes,
                textcoords=ax.transAxes,
                **arrows[vi],
            )

def update(frame):
    updated_pr_quads = []
    for sampleidx in range(nsamples):
        if sampleidx != 1:
            continue
        ax = axd[f"pred_pr {sampleidx}"]
        pr_quad = plot_map(pred_da.isel(time=frame, sample_id=0), ax=ax, style="pr_hourly")
        updated_pr_quads.append(pr_quad)
    
    return updated_pr_quads

anim = animation.FuncAnimation(fig, update, frames=nframes, blit=False)
plt.close(fig)  # prevent double display in notebook
# Display as HTML5 video (works in JupyterLab)
HTML(anim.to_html5_video())

# plt.show()

1.0
